# DeltaConv : parcours de reproduction officielle

**État : recette préparée, installation et inférence non exécutées dans l'environnement de travail actuel.**
Le code et les poids viennent des auteurs ; ce notebook ne réimplémente pas leur réseau.

Ce parcours requiert Linux x86_64 avec accès réseau ; Colab convient comme cible à vérifier. L'environnement
historique Python 3.9 / PyTorch 1.11 est créé séparément du noyau du notebook. Le solveur est fixé et son
empreinte vérifiée. Le paquet DeltaConv est ensuite réinstallé depuis le commit fixé, y compris son extension
native : CMake et un compilateur C++ doivent être disponibles. Les dépendances transitives résolues sont enregistrées, mais leur disponibilité future n'est
pas garantie. Le téléchargement ModelNet40 et son prétraitement prennent bien plus de temps que l'audit CPU.

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nassim-arifette/deltaconv-point-clouds.git"
here = Path.cwd()
project = next((p for p in (here, here.parent) if (p / "deltaconv_audit").is_dir()), None)
if project is None:
    if not IN_COLAB:
        raise RuntimeError("Run this notebook from the project or its notebooks directory.")
    project = Path("/content/deltaconv-point-clouds")
    if not project.exists():
        subprocess.run(["git", "clone", REPO_URL, str(project)], check=True)
    if not (project / "deltaconv_audit").is_dir():
        raise RuntimeError("The published project checkout is missing. Upload the project folder instead.")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
print("Project:", project)

In [ ]:
work = project / "work/official"
subprocess.run([sys.executable, str(project / "scripts/setup_official.py"),
                "--workdir", str(work), "--install-env"], check=True)
official_python = work / "environment/bin/python"
official_repo = work / "DeltaConv"

## Vérifier la parité de la baseline

Ce contrôle appelle le gradient PyTorch officiel et la transcription NumPy avec les mêmes points, voisins,
bases et signaux, en float64. Les erreurs doivent rester sous `1e-8`. Un échec bloque l'attribution au code
officiel des mesures de la transcription. Aucun fichier de succès n'est fourni d'avance.

In [ ]:
subprocess.run([str(official_python), str(project / "scripts/check_upstream.py"),
                "--upstream", str(official_repo)], check=True)

## Évaluer le checkpoint officiel sur ModelNet40

La commande suivante est celle du script officiel de reproduction, avec son checkpoint, et enregistre sa
sortie. Elle peut télécharger le jeu de données et traiter aussi les maillages d'entraînement lors de la
préparation. Les résultats qu'elle affichera seront les seuls résultats de classification de cette exécution.
En cas de panne du serveur de données historique, conserver l'erreur et installer les données officielles
manuellement ; ne pas substituer silencieusement un autre dataset.

In [ ]:
import hashlib, json
checkpoint = official_repo / "experiments/pretrained_weights/modelnet40.pt"
if not checkpoint.is_file():
    raise FileNotFoundError(checkpoint)
command = [str(official_python), "experiments/train_modelnet.py", "--checkpoint", str(checkpoint),
           "--seed", "1", "--batch_size", "8", "--num_points", "1024", "--k", "20"]
log = work / "modelnet40-evaluation.log"
receipt = {"command": command, "checkpoint_sha256": hashlib.sha256(checkpoint.read_bytes()).hexdigest(),
           "status": "running"}
(work / "modelnet40-run.json").write_text(json.dumps(receipt, indent=2))
with log.open("w", encoding="utf-8") as handle:
    result = subprocess.run(command, cwd=official_repo, stdout=handle, stderr=subprocess.STDOUT)
receipt.update(status="ok" if result.returncode == 0 else "failed", returncode=result.returncode,
               log_sha256=hashlib.sha256(log.read_bytes()).hexdigest())
(work / "modelnet40-run.json").write_text(json.dumps(receipt, indent=2))
print(log.read_text(encoding="utf-8")[-6000:])
result.check_returncode()

## Étape suivante, distincte des résultats actuels

Pour mesurer la robustesse du réseau : figer un sous-ensemble de test et des nuages prétraités avec leurs
empreintes, conserver le checkpoint, puis appliquer des perturbations appariées. Modifier les opérateurs
d'un checkpoint préentraîné change sa distribution d'activation ; comparer précision et invariance séparément.
Ne pas convertir les résultats analytiques de l'audit CPU en taux de réussite sur ModelNet40.